In [2]:
import math
import copy

class TicTacToe:
    def __init__(self):
        self.board = [[' ' for _ in range(3)] for _ in range(3)]
        self.current_player = 'X'
        self.ai_player = 'O'
        self.human_player = 'X'
        self.game_over = False
        self.winner = None

    def display_board(self):
        print("\nCurrent Board:")
        print("   0   1   2")
        for i in range(3):
            print(f"{i}  {self.board[i][0]} | {self.board[i][1]} | {self.board[i][2]}")
            if i < 2:
                print("  ---|---|---")
        print()

    def is_valid_move(self, row, col):
        return (0 <= row < 3 and 0 <= col < 3 and
                self.board[row][col] == ' ')

    def make_move(self, row, col, player):
        if self.is_valid_move(row, col):
            self.board[row][col] = player
            return True
        return False

    def undo_move(self, row, col):
        self.board[row][col] = ' '

    def check_winner(self):

        for row in self.board:
            if row[0] == row[1] == row[2] != ' ':
                return row[0]

        for col in range(3):
            if self.board[0][col] == self.board[1][col] == self.board[2][col] != ' ':
                return self.board[0][col]

        if self.board[0][0] == self.board[1][1] == self.board[2][2] != ' ':
            return self.board[0][0]
        if self.board[0][2] == self.board[1][1] == self.board[2][0] != ' ':
            return self.board[0][2]

        if all(self.board[i][j] != ' ' for i in range(3) for j in range(3)):
            return 'Draw'

        return None

    def get_available_moves(self):
        moves = []
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == ' ':
                    moves.append((i, j))
        return moves

    def evaluate_board(self):

        winner = self.check_winner()
        if winner == self.ai_player:
            return 10
        elif winner == self.human_player:
            return -10
        else:
            return 0

    def minimax(self, depth, is_maximizing, alpha=float('-inf'), beta=float('inf'), max_depth=9):

        score = self.evaluate_board()
        if score == 10 or score == -10:
            return score - depth if score == 10 else score + depth

        if depth >= max_depth or not self.get_available_moves():
            return 0

        if is_maximizing:
            max_eval = float('-inf')
            for row, col in self.get_available_moves():
                self.make_move(row, col, self.ai_player)
                eval_score = self.minimax(depth + 1, False, alpha, beta, max_depth)
                self.undo_move(row, col)
                max_eval = max(max_eval, eval_score)
                alpha = max(alpha, eval_score)
                if beta <= alpha:
                    break
            return max_eval
        else:
            min_eval = float('inf')
            for row, col in self.get_available_moves():
                self.make_move(row, col, self.human_player)
                eval_score = self.minimax(depth + 1, True, alpha, beta, max_depth)
                self.undo_move(row, col)
                min_eval = min(min_eval, eval_score)
                beta = min(beta, eval_score)
                if beta <= alpha:
                    break
            return min_eval

    def get_best_move(self, player, depth=9):
        best_move = None
        best_value = float('-inf') if player == self.ai_player else float('inf')

        for row, col in self.get_available_moves():
            self.make_move(row, col, player)

            if player == self.ai_player:
                move_value = self.minimax(0, False, float('-inf'), float('inf'), depth)
                if move_value > best_value:
                    best_value = move_value
                    best_move = (row, col)
            else:
                move_value = self.minimax(0, True, float('-inf'), float('inf'), depth)
                if move_value < best_value:
                    best_value = move_value
                    best_move = (row, col)

            self.undo_move(row, col)

        return best_move

    def get_human_move(self):
        while True:
            try:
                move = input(f"Player {self.current_player}, enter your move (row col): ").strip()
                if move.lower() == 'quit':
                    return None

                row, col = map(int, move.split())
                if self.is_valid_move(row, col):
                    return (row, col)
                else:
                    print("Invalid move! Position already taken or out of bounds.")
            except (ValueError, IndexError):
                print("Invalid input! Please enter row and column numbers (0-2) separated by space.")

    def play_human_vs_computer(self):
        print("=== TIC TAC TOE: HUMAN VS COMPUTER ===")
        print("You are 'X', Computer is 'O'")
        print("Enter moves as 'row col' (0-2), or 'quit' to exit")

        while True:
            try:
                depth = int(input("Enter AI difficulty (1-9, higher = harder): "))
                if 1 <= depth <= 9:
                    break
                else:
                    print("Please enter a number between 1 and 9.")
            except ValueError:
                print("Please enter a valid number.")

        self.display_board()

        while not self.game_over:
            if self.current_player == self.human_player:
                print(f"Your turn ({self.current_player}):")
                move = self.get_human_move()
                if move is None:
                    print("Game quit by player.")
                    return

                row, col = move
                self.make_move(row, col, self.current_player)

            else:
                print(f"Computer's turn ({self.current_player}):")
                row, col = self.get_best_move(self.ai_player, depth)
                self.make_move(row, col, self.current_player)
                print(f"Computer plays: {row} {col}")

            self.display_board()

            winner = self.check_winner()
            if winner:
                self.game_over = True
                if winner == 'Draw':
                    print("It's a draw!")
                elif winner == self.human_player:
                    print("You win! Congratulations!")
                else:
                    print("Computer wins!")
                return

            # Switch players
            self.current_player = self.ai_player if self.current_player == self.human_player else self.human_player

    def play_computer_vs_computer(self):
        print("=== TIC TAC TOE: COMPUTER VS COMPUTER ===")
        print("Computer 1 is 'X', Computer 2 is 'O'")

        while True:
            try:
                depth1 = int(input("Enter Computer 1 (X) difficulty (1-9): "))
                depth2 = int(input("Enter Computer 2 (O) difficulty (1-9): "))
                if 1 <= depth1 <= 9 and 1 <= depth2 <= 9:
                    break
                else:
                    print("Please enter numbers between 1 and 9.")
            except ValueError:
                print("Please enter valid numbers.")

        self.display_board()
        move_count = 0

        while not self.game_over:
            current_depth = depth1 if self.current_player == 'X' else depth2
            current_ai = self.current_player

            print(f"Computer {1 if self.current_player == 'X' else 2}'s turn ({self.current_player}):")


            if self.current_player == 'X':
                temp_ai = self.ai_player
                self.ai_player = 'X'
                self.human_player = 'O'
                row, col = self.get_best_move('X', current_depth)
                self.ai_player = temp_ai
                self.human_player = 'X'
            else:
                row, col = self.get_best_move('O', current_depth)

            self.make_move(row, col, self.current_player)
            print(f"Computer {1 if self.current_player == 'X' else 2} plays: {row} {col}")

            self.display_board()
            move_count += 1

            input("Press Enter to continue...")

            winner = self.check_winner()
            if winner:
                self.game_over = True
                if winner == 'Draw':
                    print("It's a draw!")
                elif winner == 'X':
                    print("Computer 1 (X) wins!")
                else:
                    print("Computer 2 (O) wins!")
                return

            # Switch players
            self.current_player = 'O' if self.current_player == 'X' else 'X'


def main():
    while True:
        print("\n" + "="*40)
        print("    TIC TAC TOE WITH MINIMAX AI")
        print("="*40)
        print("1. Human vs Computer")
        print("2. Computer vs Computer")
        print("3. Exit")
        print("="*40)

        choice = input("Select game mode (1-3): ").strip()

        if choice == '1':
            game = TicTacToe()
            game.play_human_vs_computer()
        elif choice == '2':
            game = TicTacToe()
            game.play_computer_vs_computer()
        elif choice == '3':
            print("Thank you for playing!")
            break
        else:
            print("Invalid choice! Please enter 1, 2, or 3.")

        if choice in ['1', '2']:
            play_again = input("\nWould you like to play again? (y/n): ").strip().lower()
            if play_again != 'y' and play_again != 'yes':
                print("Thank you for playing!")
                break


if __name__ == "__main__":
    main()


    TIC TAC TOE WITH MINIMAX AI
1. Human vs Computer
2. Computer vs Computer
3. Exit
Select game mode (1-3): 1
=== TIC TAC TOE: HUMAN VS COMPUTER ===
You are 'X', Computer is 'O'
Enter moves as 'row col' (0-2), or 'quit' to exit
Enter AI difficulty (1-9, higher = harder): 1

Current Board:
   0   1   2
0    |   |  
  ---|---|---
1    |   |  
  ---|---|---
2    |   |  

Your turn (X):
Player X, enter your move (row col): 0
Invalid input! Please enter row and column numbers (0-2) separated by space.
Player X, enter your move (row col): 0 0

Current Board:
   0   1   2
0  X |   |  
  ---|---|---
1    |   |  
  ---|---|---
2    |   |  

Computer's turn (O):
Computer plays: 0 1

Current Board:
   0   1   2
0  X | O |  
  ---|---|---
1    |   |  
  ---|---|---
2    |   |  

Your turn (X):
Player X, enter your move (row col): 1 1

Current Board:
   0   1   2
0  X | O |  
  ---|---|---
1    | X |  
  ---|---|---
2    |   |  

Computer's turn (O):
Computer plays: 2 2

Current Board:
   0   1  